# Multimodal emotion recognition on TESS: run everything on Colab
Use **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 1. Project files
Upload `multimodal-emotion-tess.zip` (folder icon on the left), then run:

In [ ]:
!unzip -q -o multimodal-emotion-tess.zip
%cd multimodal-emotion-tess
!pip install -q -r requirements.txt

## 2. Get TESS
The Kaggle download works with `kagglehub` (needs your Kaggle username + API token). The data set contains every clip twice; `prepare_data.py` keeps one copy.

In [ ]:
import kagglehub
kagglehub.login()
path = kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess")
print(path)

*Alternative:* if you already have `tess.zip` (e.g. in Google Drive), use this instead of the cell above:

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q -o /content/drive/MyDrive/tess.zip -d /content/tess
# path = "/content/tess"

In [ ]:
!python common/prepare_data.py --tess_dir "{path}" --out data/metadata.csv

Expected: **2800 clips, 2 speakers, 200 words**, 1960 / 420 / 420 in train / val / test, and 0 words shared between train and test.

## 3. Quick check (one seed, speech model)

In [ ]:
!python models/speech_pipeline/train.py --seed 0
!python models/speech_pipeline/test.py --seed 0

## 4. Main experiment: 4 models x 3 seeds, then the summary tables

In [ ]:
!bash run_all.sh 0 1 2

In [ ]:
!cat Results/accuracy_table.md
!cat Results/per_class_f1.md
!cat Results/silhouette_table.md

In [ ]:
!zip -qr results.zip Results checkpoints data/metadata.csv
from google.colab import files
files.download("results.zip")

## 5. Optional extra experiments (results go to separate folders, so the main results are not overwritten)
### 5a. Speaker-held-out test (train on one speaker, test on the other): a much harder test than the word split

In [ ]:
!python common/prepare_data.py --tess_dir "{path}" --out data/metadata_speaker.csv --split_mode speaker --test_speaker YAF
OPT = "--meta data/metadata_speaker.csv --results_dir Results_speaker --ckpt_dir checkpoints_speaker"
!python models/speech_pipeline/train.py --seed 0 {OPT}
!python models/speech_pipeline/test.py  --seed 0 {OPT}
!python models/fusion_pipeline/train.py --seed 0 --fusion concat {OPT}
!python models/fusion_pipeline/test.py  --seed 0 --fusion concat {OPT}
!python models/text_pipeline/train.py --seed 0 {OPT}
!python models/text_pipeline/test.py  --seed 0 {OPT}
!cat Results_speaker/*/seed0/metrics.json | grep -E '"accuracy"|"macro_f1"'

### 5b. Text branch with DistilBERT token features instead of trainable word embeddings

In [ ]:
OPT = "--results_dir Results_bert --ckpt_dir checkpoints_bert"
!python models/text_pipeline/train.py --seed 0 --text_features bert {OPT}
!python models/text_pipeline/test.py  --seed 0 {OPT}
!python models/fusion_pipeline/train.py --seed 0 --fusion concat --text_features bert {OPT}
!python models/fusion_pipeline/test.py  --seed 0 --fusion concat {OPT}